# Hybrid Neural Compute — input adapters, modules and recurring cycles

This notebook extends the Neural Network Simulator with CSV/JSON/PDF-text/blob ingestion, user-controlled algorithmic units, supervised retraining, and a safe policy unit. A computational unit is not literally a CNN neuron. The hash encoder is a wiring demo, **not a semantic encoder**.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root / "src"))
import numpy as np, torch
from neural_network_simulator.runtime import HybridRuntime, ingest_blob, ingest_path, AgentPolicyUnit, ExternalPredictionUnit
from neural_network_simulator.core import AlgorithmicNetwork
print("Torch:", torch.__version__)

## 1. Ingestion: different formats become a consistent tensor
Use modality-specific encoders in real systems. Here, a deterministic byte-hashing adapter illustrates the interface.

In [ ]:
csv_file = root / "examples" / "sample.csv"
json_file = root / "examples" / "sample.json"
records = [ingest_path(csv_file, feature_dim=16), ingest_path(json_file, feature_dim=16), ingest_blob(b"binary object example", feature_dim=16)]
[(r.kind, r.features.shape, r.detail) for r in records]

## 2. Configurable neural units
Change `channels` from 1 to 64. Internally each built-in algorithm produces a trainable scalar feature; fusion is learned.

In [ ]:
channels, features = 12, 16
runtime = HybridRuntime(channels=channels, feature_dim=features)
x = torch.cat([r.features for r in records], dim=0)
print("module tensor", runtime.model.unit_outputs(x).shape)
print("fusion logits", runtime.model(x).shape)

## 3. Two processing cycles, with optional *supervised* retraining
These labels are arbitrary demonstration labels, not verified ground truth. Repeated inference without labels does not retrain a model.

In [ ]:
first = runtime.run_cycle([csv_file, json_file])
second = runtime.run_cycle([csv_file, json_file], labels=[0, 1], epochs=10)
print(first)
print(second)

## 4. Program your own neuron-style Python module
A custom module can implement differentiable math. External sklearn/LLM calls can be wired in but are non-differentiable unless specialized training is implemented.

In [ ]:
from torch import nn
class CustomSine(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(dim)*0.05)
    def forward(self, x):
        return torch.sin(x @ self.w)
unit = CustomSine(features)
network = AlgorithmicNetwork(features, 2, custom_units=[unit, AgentPolicyUnit()])
print(network(x))

## 5. External model adapter (mocked prediction function)
A real adapter may call a fitted sklearn model or a separately hosted LLM API. Here we keep this runnable offline.

In [ ]:
external = ExternalPredictionUnit(lambda batch: np.mean(batch, axis=1))
network = AlgorithmicNetwork(features, 2, custom_units=[CustomSine(features), external])
print(network(x))

## 6. Visualize module outputs
Optional Seaborn plot; install via the notebook extras.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
with torch.no_grad():
    values = runtime.model.unit_outputs(x).numpy()
plt.figure(figsize=(10,3))
sns.heatmap(values, cmap="viridis", annot=False)
plt.xlabel("Computational unit"); plt.ylabel("Input record"); plt.title("Module output tensor"); plt.show()

## 7. Production boundaries
- Scanned PDF: add OCR. Images: add vision encoder. CSV: use schema-aware per-row feature extraction.
- Temporal, Argo or Kubernetes can trigger finite idempotent cycles with durable queues, retries and checkpoints. `run_cycle` does not do this automatically.
- Train only on validated labeled data, with holdout evaluations, drift gates, approvals and registry versioning.
- LLM tool use should run in permissioned agents with sandboxed actions; model output is not trusted executable code.
- Rust via PyO3 is a possible hot-path optimization; profile before adding it.